In [1]:
import os
from pyspark.sql.functions import regexp_replace
# Set JAVA_HOME to the path of Java 17
# (This command finds the path dynamically using the mac system tool)
java17_path = os.popen("/usr/libexec/java_home -v 17").read().strip()

if java17_path:
    os.environ["JAVA_HOME"] = java17_path
    print(f"Successfully set JAVA_HOME to: {java17_path}")
else:
    print("Error: Java 17 not found! Please verify installation.")

Successfully set JAVA_HOME to: /Library/Java/JavaVirtualMachines/jdk-17.jdk/Contents/Home


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Data Aggregation") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/01/01 05:48:02 WARN Utils: Your hostname, Mushtaq.local, resolves to a loopback address: 127.0.0.1; using 192.168.68.66 instead (on interface en0)
26/01/01 05:48:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/01 05:48:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
os.environ.get('JAVA_HOME')

'/Library/Java/JavaVirtualMachines/jdk-17.jdk/Contents/Home'

In [4]:
listings = spark.read.csv("../data/raw/listings.csv.gz", 
    header=True,
    inferSchema=True,
    sep=",",
    quote='"',
    escape='"',
    multiLine=True,
    mode="PERMISSIVE"
)

In [5]:
listings \
  .groupby(listings.property_type) \
  .count() \
  .show(truncate=False)

+----------------------------------+-----+
|property_type                     |count|
+----------------------------------+-----+
|Private room in lighthouse        |2    |
|Private room in loft              |153  |
|Private room in earthen home      |2    |
|Entire chalet                     |4    |
|Earthen home                      |1    |
|Farm stay                         |4    |
|Entire rental unit                |41215|
|Shared room in hostel             |66   |
|Shared room                       |1    |
|Private room in condo             |3189 |
|Room in boutique hotel            |217  |
|Private room in religious building|4    |
|Room in bed and breakfast         |18   |
|Private room in casa particular   |56   |
|Private room in bungalow          |63   |
|Entire cabin                      |43   |
|Entire guesthouse                 |221  |
|Hut                               |3    |
|Private room in nature lodge      |4    |
|Entire guest suite                |177  |
+----------

In [6]:
import pyspark.sql.functions as F

listings \
  .groupby(listings.property_type) \
  .agg(
    F.count('property_type').alias('count')
  ) \
  .orderBy('count', ascending=[False]) \
  .show(truncate=False)

+----------------------------------+-----+
|property_type                     |count|
+----------------------------------+-----+
|Entire rental unit                |41215|
|Private room in rental unit       |14464|
|Private room in home              |11704|
|Entire home                       |9120 |
|Entire condo                      |8250 |
|Private room in condo             |3189 |
|Entire serviced apartment         |1874 |
|Private room in townhouse         |1195 |
|Room in hotel                     |1113 |
|Entire townhouse                  |1058 |
|Private room in bed and breakfast |491  |
|Private room in guesthouse        |377  |
|Entire loft                       |341  |
|Entire guesthouse                 |221  |
|Room in boutique hotel            |217  |
|Entire guest suite                |177  |
|Private room in guest suite       |174  |
|Private room in loft              |153  |
|Private room in serviced apartment|132  |
|Private room                      |103  |
+----------

In [7]:

listings \
  .groupby(listings.property_type) \
  .agg(
    F.count('property_type').alias('count'),
    F.avg('review_scores_location')
  ) \
  .orderBy('count', ascending=[False]) \
  .show(truncate=False)

+----------------------------------+-----+---------------------------+
|property_type                     |count|avg(review_scores_location)|
+----------------------------------+-----+---------------------------+
|Entire rental unit                |41215|4.727437046043664          |
|Private room in rental unit       |14464|4.726647667299953          |
|Private room in home              |11704|4.694735943407702          |
|Entire home                       |9120 |4.727462068965489          |
|Entire condo                      |8250 |4.770284788770394          |
|Private room in condo             |3189 |4.778348954578206          |
|Entire serviced apartment         |1874 |4.722610024449879          |
|Private room in townhouse         |1195 |4.766042471042479          |
|Room in hotel                     |1113 |4.608689075630254          |
|Entire townhouse                  |1058 |4.81762931034483           |
|Private room in bed and breakfast |491  |4.724728915662653          |
|Priva

In [9]:
reviews = spark.read.csv("../data/raw/reviews.csv.gz", 
    header=True,
    inferSchema=True,
    sep=",",
    quote='"',
    escape='"',
    multiLine=True,
    mode="PERMISSIVE"
)

In [10]:
for field in reviews.schema:
    print(field)

StructField('listing_id', LongType(), True)
StructField('id', LongType(), True)
StructField('date', DateType(), True)
StructField('reviewer_id', IntegerType(), True)
StructField('reviewer_name', StringType(), True)
StructField('comments', StringType(), True)


In [11]:
listings_reviews = listings.join(
    reviews, listings.id == reviews.listing_id, how='inner'
)

In [15]:
listings_reviews \
  .groupBy(listings.id) \
  .agg(
    F.count(reviews.id).alias('num_reviews')
  ) \
  .show()

+-------+-----------+
|     id|num_reviews|
+-------+-----------+
|  78606|          2|
| 444886|         12|
| 466017|         28|
| 991477|          5|
|2557853|         92|
|2736493|          4|
|3132302|          3|
|3734796|          5|
|3997029|          7|
|3917692|          1|
|4361078|         71|
|5355817|         18|
|5520243|          6|
|5921026|         40|
|6311069|          3|
|6552071|          1|
|6651481|         48|
|6606418|        262|
|7188835|        206|
|7709953|         34|
+-------+-----------+
only showing top 20 rows


In [29]:
reviews_per_listing = listings_reviews \
  .groupBy(listings.id, listings.name) \
  .agg(
    F.count(reviews.id).alias('num_reviews')
  ) \
  .orderBy('num_reviews', ascending=False) \
  .limit(10) \
  .show(truncate=False)

+--------+--------------------------------------------------+-----------+
|id      |name                                              |num_reviews|
+--------+--------------------------------------------------+-----------+
|47408549|Double Room+ Ensuite                              |1902       |
|43120947|Private double room with en suite facilities      |1647       |
|19670926|Locke Studio Apartment at Leman Locke             |1443       |
|2126708 |London's best transport hub 5 mins walk! Safe too!|1142       |
|46233904|Superior Studio, avg size 23.5 msq                |1002       |
|2659707 |Large Room + Private Bathroom, E3.                |998        |
|27833488|S - Heathrow Airport Terminal 2 3 4 5 Hatton Cross|951        |
|4748665 |Single bedroom near London Stratford              |933        |
|42081759|Micro Studio at Locke at Broken Wharf             |914        |
|5266466 |Large London Room, Ensuite Bathroom,TV & Breakfast|909        |
+--------+----------------------------

In [28]:
listings.groupBy(listings.host_name).agg(F.count(listings.id).alias('count'),F.avg(listings.review_scores_rating).alias('Average_Review')).show()

+--------------+-----+------------------+
|     host_name|count|    Average_Review|
+--------------+-----+------------------+
|      Francois|    8|              4.66|
|        Britta|    3|             4.925|
|        Orphea|    1|               4.0|
|Maria & Graham|    2|              4.94|
|       Susanna|   15| 4.836666666666667|
|          Nell|    4|             4.835|
|          Saad|    4|               4.0|
|      Laurence|   22|4.8088235294117645|
|        Hideki|    1|               4.0|
|        Hantao|    1|               5.0|
|      Julianne|    4|               4.0|
|      Haritini|    1|              4.85|
|          Faye|   12|             4.699|
|     Amaryllis|    2|              3.33|
|         Tyler|    8| 4.886666666666667|
|             K|   22| 4.641428571428571|
|   Jean-Michel|    2|              NULL|
|    Andy & Fee|    1|               5.0|
|          Hina|    6|              4.58|
|        Dougie|    2|4.6899999999999995|
+--------------+-----+------------

In [ ]:
listings_reviews